# DSA 210 — Exploratory Data Analysis (EDA)
## Predicting Agricultural Crop Yield Across Turkish Provinces

This notebook contains all exploratory data analysis visualizations for the project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 12, "figure.dpi": 120, "savefig.bbox": "tight"})
sns.set_theme(style="whitegrid", palette="colorblind")

df = pd.read_csv("../DATA/turkey_agriculture_dataset.csv")
print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

## 1. Target Variable Distribution (Class Balance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_counts = df["verim_sinifi_label"].value_counts()
colors = ["#2ecc71", "#e74c3c"]

class_counts.plot(kind="bar", ax=axes[0], color=colors, edgecolor="black", alpha=0.85)
axes[0].set_title("Target Variable Distribution\n(Yield Class: High vs Low)", fontsize=14)
axes[0].set_xlabel("Yield Class")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 50, str(v), ha="center", fontweight="bold")

axes[1].pie(class_counts.values, labels=class_counts.index, colors=colors,
            autopct="%1.1f%%", startangle=90, textprops={"fontsize": 13})
axes[1].set_title("Class Proportions")
plt.tight_layout()
plt.show()

print(f"High yield: {class_counts.get('Yüksek', 0)} | Low yield: {class_counts.get('Düşük', 0)}")
print(f"→ Classes are well-balanced — no need for SMOTE or undersampling.")

## 2. Yield Distribution by Crop Type

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
crops = df["urun"].unique()

for i, crop in enumerate(crops):
    ax = axes[i // 2][i % 2]
    crop_data = df[df["urun"] == crop]["verim_kg_dekar"]
    ax.hist(crop_data, bins=40, color=sns.color_palette("colorblind")[i], edgecolor="black", alpha=0.75)
    ax.axvline(crop_data.median(), color="red", linestyle="--", linewidth=2, label=f"Median: {crop_data.median():.0f}")
    ax.set_title(f"{crop} — Yield Distribution (kg/decar)")
    ax.set_xlabel("Yield (kg/decar)")
    ax.set_ylabel("Frequency")
    ax.legend()

plt.suptitle("Yield Distributions by Crop Type", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3. Yield Comparison by Region (Boxplot)

In [ ]:
region_order = ["Marmara", "Ege", "Akdeniz", "İç Anadolu", "Karadeniz", "Doğu Anadolu", "Güneydoğu Anadolu"]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, crop in enumerate(crops):
    ax = axes[i // 2][i % 2]
    sns.boxplot(data=df[df["urun"] == crop], x="bolge", y="verim_kg_dekar", order=region_order, ax=ax, palette="Set2")
    ax.set_title(f"{crop} — Yield by Region")
    ax.set_xlabel("")
    ax.set_ylabel("Yield (kg/decar)")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Crop Yields by Geographical Region", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Print regional means for wheat
wheat_means = df[df["urun"] == "Buğday"].groupby("bolge")["verim_kg_dekar"].mean().sort_values(ascending=False)
print("Wheat yield by region (avg kg/decar):")
for region, val in wheat_means.items():
    print(f"  {region}: {val:.1f}")

## 4. Weather Variables Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
weather_cols = [("ort_sicaklik_C", "Avg. Temperature (°C)"), ("toplam_yagis_mm", "Total Rainfall (mm)"),
                ("bagil_nem_pct", "Relative Humidity (%)"), ("kuraklik_indeksi", "Drought Index")]

for i, (col, label) in enumerate(weather_cols):
    ax = axes[i // 2][i % 2]
    prov_data = df.groupby(["il", "yil"])[col].first().reset_index()
    ax.hist(prov_data[col], bins=35, color=sns.color_palette("coolwarm", 4)[i], edgecolor="black", alpha=0.75)
    ax.set_title(f"{label} Distribution")
    ax.set_xlabel(label)
    ax.set_ylabel("Frequency")

plt.suptitle("Meteorological Variables Distributions (Province-Year Level)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
numeric_cols = ["verim_kg_dekar", "ort_sicaklik_C", "toplam_yagis_mm", "yagisli_gun",
                "bagil_nem_pct", "kuraklik_indeksi", "yagis_sapma_zscore", "sicaklik_genlik",
                "hasat_orani", "verim_degisim_pct"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix — Numeric Variables", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

print("Top correlations with yield:")
for col, val in corr["verim_kg_dekar"].drop("verim_kg_dekar").abs().sort_values(ascending=False).head(5).items():
    sign = "+" if corr.loc["verim_kg_dekar", col] > 0 else "-"
    print(f"  {col}: {sign}{val:.3f}")

## 6. Weather vs Yield Scatter Plots (Wheat)

In [ ]:
wheat = df[df["urun"] == "Buğday"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(wheat["ort_sicaklik_C"], wheat["verim_kg_dekar"], alpha=0.3, c=wheat["verim_sinifi"], cmap="RdYlGn", edgecolors="gray", s=30)
axes[0].set_xlabel("Avg. Temperature (°C)"); axes[0].set_ylabel("Wheat Yield (kg/decar)"); axes[0].set_title("Temperature vs Wheat Yield")

axes[1].scatter(wheat["toplam_yagis_mm"], wheat["verim_kg_dekar"], alpha=0.3, c=wheat["verim_sinifi"], cmap="RdYlGn", edgecolors="gray", s=30)
axes[1].set_xlabel("Total Rainfall (mm)"); axes[1].set_ylabel("Wheat Yield (kg/decar)"); axes[1].set_title("Rainfall vs Wheat Yield")

axes[2].scatter(wheat["kuraklik_indeksi"], wheat["verim_kg_dekar"], alpha=0.3, c=wheat["verim_sinifi"], cmap="RdYlGn", edgecolors="gray", s=30)
axes[2].set_xlabel("Drought Index"); axes[2].set_ylabel("Wheat Yield (kg/decar)"); axes[2].set_title("Drought Index vs Wheat Yield")

plt.suptitle("Wheat: Weather Variables vs Yield (Green=High, Red=Low)", fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

## 7. Yearly Yield Trends

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for crop in crops:
    yearly = df[df["urun"] == crop].groupby("yil")["verim_kg_dekar"].mean()
    ax.plot(yearly.index, yearly.values, marker="o", linewidth=2, label=crop)

ax.set_title("Turkey — Average Yield Trends Over Years", fontsize=15, fontweight="bold")
ax.set_xlabel("Year"); ax.set_ylabel("Avg. Yield (kg/decar)")
ax.legend(title="Crop"); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Region × Year Yield Heatmap (Wheat)

In [ ]:
pivot = df[df["urun"] == "Buğday"].pivot_table(values="verim_kg_dekar", index="bolge", columns="yil", aggfunc="mean")
pivot = pivot.reindex(region_order)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlGn", linewidths=0.5, ax=ax)
ax.set_title("Wheat — Region × Year Average Yield (kg/decar)", fontsize=15, fontweight="bold")
ax.set_xlabel("Year"); ax.set_ylabel("Region")
plt.tight_layout()
plt.show()